# Chinor Bank Assistant: run a 14B adapter on Colab (draft)

**4-bit on a T4, not the evaluation protocol; answers can differ from the recorded ones.**

**Untested draft.** Written without a GPU and before any Hugging Face Hub repository exists: it has never been run.
The adapter locations are placeholders marked `TODO-HUB`; the cells that download an adapter fail until they are
filled. The recorded answers (Protocol A: H200, bf16, eval batch 32) are in `eval/outputs/` and in the
recorded-answer viewer; this notebook loads `Qwen/Qwen3-14B-Base` in 4-bit NF4 instead, so its greedy answers are a
different run.

Chinor Bank is fictional: every policy, fee and number the model knows was invented for the project.

What it does, as `demo/app.py --4bit`: the base model in 4-bit, one adapter chosen from the list below (its trained
`<|im_start|>` / `<|im_end|>` rows are part of the adapter), the project template `train/chat_template.jinja`, the
frozen system prompt `handbook/system_prompt.txt`, greedy decoding with the stop tokens and token limit of
`demo/app.py`. The base revision is read from `docs/base_revisions.md` and the adapter file is checked against
`results/adapter_sha256.tsv`. No token or secret is needed: the repository and the base model are public.

In [ ]:
# Parameters
REPO = "https://github.com/Sarvarbek-Erniyazov/chinor-bank-assistant-public.git"
REF = "main"   # TODO-TAG: pin the v1.2.0 release tag once it exists
ADAPTER = "l14b_sft_rslora_tok-s0-v1.2"   # one of ADAPTERS below

In [ ]:
import os, subprocess
if not os.path.isdir("repo"):
    subprocess.run(["git", "clone", "-q", REPO, "repo"], check=True)
%cd repo
!git checkout -q {REF}
!git log --oneline -1

In [ ]:
# torch is not pinned: Colab's preinstalled torch is kept (requirements-gpu.txt explains why)
!pip install -q -r requirements-gpu.txt
!pip uninstall -y -q torchvision torchaudio
import torch, transformers, peft, bitsandbytes
print(torch.__version__, transformers.__version__, peft.__version__, bitsandbytes.__version__,
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU")

In [ ]:
# Adapters that load directly on Qwen3-14B-Base. Each value is the Hub repository the adapter will be uploaded to.
# TODO-HUB: fill in when the adapters are on the Hub (model_cards/ say "upload pending").
# DPO / SimPO adapters are not listed: they load on top of sft-merged (the SFT adapter merged into the base), which
# is not on the Hub either.
ADAPTERS = {
    "l14b_sft_rslora_tok-s0-v1.2": "TODO-HUB",   # SFT rsLoRA with token rows, data v1.2, seed 0 (parent of DPO / SimPO)
    "l14b_sft_rslora_tok-s1-v1.2": "TODO-HUB",
    "l14b_sft_rslora_tok-s0": "TODO-HUB",        # SFT rsLoRA with token rows, data v1
    "l14b_qlora_rslora_tok-s0": "TODO-HUB",      # QLoRA (trained on a 4-bit base)
    "l14b_sft_rslora-s0": "TODO-HUB",            # without token rows: never emits <|im_end|>, runs to the token limit
}
assert ADAPTER in ADAPTERS, f"choose one of {list(ADAPTERS)}"

In [ ]:
import csv, hashlib, re, sys
from pathlib import Path

sys.path.insert(0, "demo")
from app import IM_END, ENDOFTEXT, MAX_NEW_TOKENS   # the decoding constants of the demo and the evaluation

BASE = "Qwen/Qwen3-14B-Base"
REVISION = next(re.split(r"\s*\|\s*", l.strip("| "))[2].strip("`")
                for l in Path("docs/base_revisions.md").read_text(encoding="utf-8").splitlines()
                if l.startswith(f"| `{BASE}`"))
SHA = {r[0]: r[1] for r in csv.reader(open("results/adapter_sha256.tsv", encoding="utf-8"), delimiter="\t")}
SYSTEM = Path("handbook/system_prompt.txt").read_text(encoding="utf-8").strip()
TEMPLATE = Path("train/chat_template.jinja").read_text(encoding="utf-8")
print(BASE, REVISION, "| adapter", ADAPTER, SHA[ADAPTER][:16] + "…", "| max new tokens", MAX_NEW_TOKENS)

In [ ]:
# Download the adapter and check it against results/adapter_sha256.tsv
from huggingface_hub import snapshot_download
hub_id = ADAPTERS[ADAPTER]
assert hub_id != "TODO-HUB", "TODO-HUB: the adapter is not on the Hub yet; set its repository id in ADAPTERS"
adapter_dir = snapshot_download(hub_id)
got = hashlib.sha256(Path(adapter_dir, "adapter_model.safetensors").read_bytes()).hexdigest()
assert got == SHA[ADAPTER], f"adapter SHA-256 {got} does not match results/adapter_sha256.tsv"
print("adapter verified:", adapter_dir)

In [ ]:
# Base model in 4-bit NF4 (as demo/app.py --4bit), then the adapter
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16,
                         bnb_4bit_use_double_quant=True)
tok = AutoTokenizer.from_pretrained(BASE, revision=REVISION)
tok.chat_template = TEMPLATE
model = AutoModelForCausalLM.from_pretrained(BASE, revision=REVISION, quantization_config=bnb, dtype=torch.float16,
                                             device_map="auto")
model = PeftModel.from_pretrained(model, adapter_dir)
model.eval()
print(f"peak VRAM {torch.cuda.max_memory_allocated() / 2**30:.1f} GiB")

In [ ]:
def generate(msgs):
    """Greedy, stop on <|im_end|> or <|endoftext|>; returns (text, stop reason, new tokens)."""
    text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    ids = tok(text, return_tensors="pt", add_special_tokens=False)["input_ids"].to(model.device)
    with torch.no_grad():
        out = model.generate(ids, do_sample=False, max_new_tokens=MAX_NEW_TOKENS, eos_token_id=[IM_END, ENDOFTEXT],
                             pad_token_id=tok.pad_token_id or ENDOFTEXT)
    new = out[0, ids.shape[1]:].tolist()
    stop = {IM_END: "im_end", ENDOFTEXT: "endoftext"}.get(new[-1], "length")
    return tok.decode(new, skip_special_tokens=False), stop, len(new)


def ask(question):
    """One free-text question with the frozen system prompt, as the live demo does."""
    return generate([{"role": "system", "content": SYSTEM}, {"role": "user", "content": question}])

reply, stop, n = ask("Can I cancel my transfer?")
print(f"[{stop}, {n} tokens]\n{reply}")

In [ ]:
# The pre-registered example items (results/qualitative_ids.json) next to the recorded Protocol-A answer of this run.
# Messages as the evaluation builds them (eval/run_eval.py), so block R items get no system prompt.
import json
sys.path.insert(0, "eval")
from run_eval import build_messages   # noqa: E402

items = {r["id"]: r for r in map(json.loads, open("eval/eval_set.jsonl", encoding="utf-8")) if "id" in r}
recorded = {r["id"]: r for r in map(json.loads, open(f"eval/outputs/{ADAPTER}/answers.jsonl", encoding="utf-8"))}
for block, item_id in json.load(open("results/qualitative_ids.json", encoding="utf-8"))["chosen"].items():
    msgs = build_messages(items[item_id], SYSTEM, [])   # zero-shot, as every trained arm was evaluated
    reply, stop, n = generate(msgs)
    print("=" * 100, f"\n{item_id} ({block}): {msgs[-1]['content']}")
    print(f"--- this notebook, 4-bit [{stop}, {n} tokens]\n{reply}")
    print(f"--- recorded, Protocol A [{recorded[item_id]['stop_reason']}]\n{recorded[item_id]['raw']}")

Untested: before relying on this notebook, check that the repository at `REF` has `docs/base_revisions.md` and
`eval/run_eval.py` `build_messages(item, system_prompt, shots)` (both are on the v1.2 branch, not yet in a release),
that the T4 fits the 4-bit 14B model with this adapter, and that the Hub ids are filled in.